# Benchmark: TabPFN vs. State-of-the-Art Algorithms

Comparison of **TabPFN** against standard ML baselines on the heart failure prediction task (3-class: early / late / healthy) using identical data pipeline (`load_final_data` → `preprocess_data` → `balance_data`).

### Algorithms
| Model | Type | Why |
|-------|------|-----|
| DummyClassifier | Baseline | Lower bound — shows what random guessing achieves |
| LogisticRegression | Linear | Classic baseline, interpretable, fast |
| RandomForest | Ensemble (Bagging) | Strong default, handles categoricals via encoding |
| XGBoost | Ensemble (Boosting) | State-of-the-art for tabular data |
| TabPFN | Foundation Model | Our main model — zero-shot Bayesian inference |

### Evaluation
- **Metrics**: Accuracy, F1 Macro, ROC-AUC (OvR), per-class F1
- **Robustness**: Each model trained on 20 balanced subsets (same seeds as TabPFN_v4)
- **Fair comparison**: Same train/val/test split, same features, same balancing

In [1]:
# Imports
import sys
sys.path.insert(0, '..')

import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, recall_score, precision_score,
    classification_report
)

from fs_thesis.data_loader import load_final_data
from fs_thesis.preprocessing import preprocess_data, balance_data, get_X_y

warnings.filterwarnings('ignore')

In [2]:
# ── Run-Ordner ──
_run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/Users/andrey/Repositories/fs-thesis/models/runs/benchmark_{_run_timestamp}")
PLOTS_DIR = RUN_DIR / "plots"
RESULTS_DIR = RUN_DIR / "results"
for d in [PLOTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_plot_counter = 0

def show_and_save(fig, name: str = None, width=1600, height=600):
    """fig.show() + PNG speichern."""
    global _plot_counter
    _plot_counter += 1
    filename = name or f"plot_{_plot_counter:02d}"
    path = PLOTS_DIR / f"{filename}.png"
    fig.write_image(str(path), scale=2, width=width, height=height)
    print(f"💾 {path}")
    fig.show()

print(f"📁 Run-Ordner: {RUN_DIR}")

📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_220721


In [3]:
import logging

log_file = RUN_DIR / "run.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)
log = logging.getLogger()
log.info(f"📁 Run-Ordner: {RUN_DIR}")


22:07:21 | 📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_220721


# 1. Data Pipeline (identical to TabPFN_v4)

In [4]:
from sklearn.model_selection import train_test_split

df = load_final_data()
df_train, df_val, df_test = preprocess_data(df)
X_val, y_val = get_X_y(df_val)
X_test, y_test = get_X_y(df_test)

_, X_val_small, _, y_val_small = train_test_split(
    X_val, y_val, test_size=3000, stratify=y_val, random_state=42
)
X_val_small = X_val_small.reset_index(drop=True)

log.info(f"Val: {len(y_val)} | Val (TabPFN subsample): {len(y_val_small)} | Test: {len(y_test)}")
log.info(f"Class distribution (val):       {np.bincount(y_val)}")
log.info(f"Class distribution (val_small): {np.bincount(y_val_small)}")

22:07:22 | Val: 35753 | Val (TabPFN subsample): 3000 | Test: 44691
22:07:22 | Class distribution (val):       [ 1719  1304 32730]
22:07:22 | Class distribution (val_small): [ 144  110 2746]


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)


# 2. Define Models

All models use the **same interface**: `fit(X_train, y_train)` → `predict(X_val)` / `predict_proba(X_val)`.

For tree-based and linear models, categorical features are **one-hot encoded** (TabPFN handles them natively). The encoding is applied inside the benchmark loop.

In [5]:
# Model Definitions
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

# Feature Types
FEATURE_COLS = [
    "gender", "anchor_age", "insurance", "language", "marital_status", "race", "admission_type", "bmi"
]
CAT_COLS = [
    "gender", "insurance", "language", "marital_status", "race", "admission_type"
]
NUM_COLS = ["anchor_age", "bmi"]

# Preprocessing Pipeline for sklearn Models
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), NUM_COLS),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CAT_COLS),
    ],
    remainder='drop'
)

# Model Dictionary
MODELS = {
    "DummyClassifier": Pipeline([
        ('prep', preprocessor),
        ('clf', DummyClassifier(strategy='stratified', random_state=42))
    ]),
    "LogisticRegression": Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42))
    ]),
    "RandomForest": Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ('prep', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            objective='multi:softprob', num_class=3,
            random_state=42, n_jobs=-1, verbosity=0,
            eval_metric='mlogloss'
        ))
    ]),
    #"TabPFN": None (extra in cell 10)
}

log.info(f"Models: {list(MODELS.keys())}")
log.info(f"Features: {len(FEATURE_COLS)} ({len(NUM_COLS)} numerical, {len(CAT_COLS)} categorical)")

22:07:23 | Models: ['DummyClassifier', 'LogisticRegression', 'RandomForest', 'XGBoost']
22:07:23 | Features: 8 (2 numerical, 6 categorical)


In [6]:
log.info(len(y_val))

22:07:23 | 35753


# 3. Robustness Benchmark Loop

Each model is trained **20 times** with different balanced training subsets (seeds 42–61), identical to the TabPFN_v4 robustness loop. This measures performance **and** stability.

In [7]:
from sklearn.base import clone

N_LOOPS = 20
N_SAMPLES = 300
log.info(f"RUN gestartet | N_LOOPS={N_LOOPS} | N_SAMPLES={N_SAMPLES}")

config = {"n_loops": N_LOOPS, "n_samples": N_SAMPLES, 
          "models": list(MODELS.keys()) + ["TabPFN", "TabICL"],  # ← TabICL hinzufügen
          "run_dir": str(RUN_DIR)}
json.dump(config, open(RUN_DIR / "config.json", "w"), indent=2)

all_results = []
start_total = time.time()

22:07:23 | RUN gestartet | N_LOOPS=20 | N_SAMPLES=300


In [8]:
import subprocess, json
tabicl_result_path = str(RESULTS_DIR / "tabicl_result.json")

log.info("Starte TabICL subprocess...")
proc_icl = subprocess.run(
    [sys.executable, "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabicl.py", tabicl_result_path],
    capture_output=False
)

if proc_icl.returncode == 0:
    tabicl_result = json.load(open(tabicl_result_path))
    all_results.append(tabicl_result)
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)
    log.info(f"✅ TabICL: F1={tabicl_result['f1_macro']:.4f}")
else:
    log.error("❌ TabICL subprocess fehlgeschlagen")

22:07:23 | Starte TabICL subprocess...


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.

predicting...
proba...
✅ F1=0.3484 | AUC=0.7951 | 22.8s


22:07:49 | ✅ TabICL: F1=0.3484


In [9]:


tabpfn_result_path = str(RESULTS_DIR / "tabpfn_result.json")

log.info("Starte TabPFN subprocess (isoliert von MPS)...")
proc = subprocess.run(
    [sys.executable, "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabpfn.py", tabpfn_result_path],
    capture_output=False
)

if proc.returncode == 0:
    tabpfn_result = json.load(open(tabpfn_result_path))
    all_results = [tabpfn_result]
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)
    log.info(f"✅ TabPFN geladen: F1={tabpfn_result['f1_macro']:.4f}")
else:
    log.error("❌ TabPFN subprocess fehlgeschlagen")
    all_results = []

22:07:49 | Starte TabPFN subprocess (isoliert von MPS)...


Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting...
proba...
✅ F1=0.3673 | AUC=0.8162 | 19.4s


22:08:13 | ✅ TabPFN geladen: F1=0.3673


In [10]:
from sklearn.base import clone

for model_name, pipeline in MODELS.items():
    log.info(f"\n{'='*60}")
    log.info(f"  {model_name}")
    log.info(f"{'='*60}")
    t0 = time.time()
    model_results = []

    for i in tqdm(range(N_LOOPS), desc=model_name):
        try:
            seed = 42 + i
            df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=seed)
            X_tr, y_tr = get_X_y(df_bal)
            clf = clone(pipeline)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_val)
            y_proba = clf.predict_proba(X_val)
            f1_pc = f1_score(y_val, y_pred, average=None)
            result = {
                'model': model_name, 'run_id': i, 'seed': seed,
                'accuracy': accuracy_score(y_val, y_pred),
                'f1_macro': f1_score(y_val, y_pred, average='macro'),
                'roc_auc_macro': roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro'),
                'recall_macro': recall_score(y_val, y_pred, average='macro'),
                'precision_macro': precision_score(y_val, y_pred, average='macro'),
                'f1_class_0_early': f1_pc[0],
                'f1_class_1_late': f1_pc[1],
                'f1_class_2_healthy': f1_pc[2],
                'time_sec': time.time() - t0
            }
            all_results.append(result)
            model_results.append(result)
        except Exception as e:
            log.error(f"  ⚠️ ERROR Run {i}: {e}")

    elapsed = time.time() - t0
    if model_results:
        df_m = pd.DataFrame(model_results)
        log.info(f"  ✅ F1={df_m['f1_macro'].mean():.4f}±{df_m['f1_macro'].std():.4f} | AUC={df_m['roc_auc_macro'].mean():.4f} | {elapsed:.1f}s")
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)

log.info(f"\n{'='*60}")
log.info(f"  DONE | {len(all_results)} total runs")
log.info(f"{'='*60}")

22:08:13 | 
22:08:13 |   DummyClassifier
22:08:13 | ============================================================


DummyClassifier:   0%|          | 0/20 [00:00<?, ?it/s]

22:08:16 |   ✅ F1=0.2106±0.0000 | AUC=0.4946 | 2.5s
22:08:16 | 
22:08:16 |   LogisticRegression
22:08:16 | ============================================================


LogisticRegression:   0%|          | 0/20 [00:00<?, ?it/s]

22:08:18 |   ✅ F1=0.3671±0.0072 | AUC=0.7693 | 2.7s
22:08:18 | 
22:08:18 |   RandomForest
22:08:18 | ============================================================


RandomForest:   0%|          | 0/20 [00:00<?, ?it/s]

22:08:26 |   ✅ F1=0.3672±0.0085 | AUC=0.7918 | 7.7s
22:08:26 | 
22:08:26 |   XGBoost
22:08:26 | ============================================================


XGBoost:   0%|          | 0/20 [00:00<?, ?it/s]

22:08:40 |   ✅ F1=0.3614±0.0069 | AUC=0.7743 | 14.2s
22:08:40 | 
22:08:40 |   DONE | 81 total runs
22:08:40 | ============================================================


# 4. Results & Comparison

In [11]:
# ── Summary Table ──
df_results = pd.DataFrame(all_results)
df_results.to_csv(RESULTS_DIR / "benchmark_all_runs.csv", index=False)

df_summary = df_results.groupby('model').agg(
    f1_mean=('f1_macro', 'mean'), f1_std=('f1_macro', 'std'),
    auc_mean=('roc_auc_macro', 'mean'), auc_std=('roc_auc_macro', 'std'),
    acc_mean=('accuracy', 'mean'), acc_std=('accuracy', 'std'),
    recall_mean=('recall_macro', 'mean'),
    precision_mean=('precision_macro', 'mean'),
    f1_early_mean=('f1_class_0_early', 'mean'), f1_early_std=('f1_class_0_early', 'std'),
    f1_late_mean=('f1_class_1_late', 'mean'), f1_late_std=('f1_class_1_late', 'std'),
    f1_healthy_mean=('f1_class_2_healthy', 'mean'), f1_healthy_std=('f1_class_2_healthy', 'std'),
    n_runs=('run_id', 'count'),
).reset_index().sort_values('f1_mean', ascending=False)

df_summary.to_csv(RESULTS_DIR / "benchmark_summary.csv", index=False)

# Schöne Darstellung
log.info("\n📊 Benchmark Summary (sorted by F1 Macro):\n")
display_cols = ['model', 'f1_mean', 'f1_std', 'auc_mean', 'auc_std', 'acc_mean', 'recall_mean', 'precision_mean']
log.info(df_summary[display_cols].to_string(index=False, float_format='{:.4f}'.format))

22:08:40 | 
📊 Benchmark Summary (sorted by F1 Macro):

22:08:40 |              model  f1_mean  f1_std  auc_mean  auc_std  acc_mean  recall_mean  precision_mean
            TabPFN   0.3673     NaN    0.8162      NaN    0.5497       0.6551          0.4069
      RandomForest   0.3672  0.0085    0.7918   0.0033    0.5659       0.6207          0.4026
LogisticRegression   0.3671  0.0072    0.7693   0.0044    0.5727       0.5863          0.4014
           XGBoost   0.3614  0.0069    0.7743   0.0057    0.5675       0.5951          0.3969
   DummyClassifier   0.2106  0.0000    0.4946   0.0000    0.3305       0.3274          0.3315


## 4.1 F1 Macro Comparison

In [12]:
# F1 Macro: Bar Chart with Error Bars
model_order = df_summary.sort_values('f1_mean')['model'].tolist()

fig = px.bar(
    df_summary.sort_values('f1_mean'),
    x='f1_mean', y='model', error_x='f1_std',
    orientation='h',
    text=df_summary.sort_values('f1_mean').apply(
        lambda r: f"{r['f1_mean']:.1%} ± {r['f1_std']:.1%}", axis=1
    ),
    title=f'Benchmark: F1 Macro ({N_LOOPS} Runs, n_samples={N_SAMPLES})',
    labels={'f1_mean': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig.update_layout(
    xaxis_tickformat='.0%',
    showlegend=False,
    yaxis=dict(categoryorder='array', categoryarray=model_order),
    height=400
)
show_and_save(fig, "benchmark_f1_macro_comparison")

22:08:40 | Chromium init'ed with kwargs {}
22:08:40 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:08:40 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpnnf430fe.
22:08:40 | Opening browser.
22:08:40 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmppqp0pza1.
22:08:40 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmppqp0pza1
22:08:41 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpnnf430fe/index.html
22:08:41 | Waiting on all navigates
22:08:41 | All navigates done, putting them all in queue.
22:08:41 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpnnf430fe/index.html
22:08:41 | Waiting on all navigates
22:08:42 | All navigates done, putting them all in queue.
22:08:42 | Tab ready: A358CFA233595C52F73847E6D9DEFB9E
22:08:42 | Getting tab from queue (has 1)
22:08:42 | Got A358
22:08:42 | Processing Benchma

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_220721/plots/benchmark_f1_macro_comparison.png


In [13]:
# F1 Macro: Violin Plot (Distribution across Runs)
fig2 = px.violin(
    df_results, x='model', y='f1_macro', box=True, points='all',
    title=f'F1 Macro Distribution ({N_LOOPS} Runs per Model)',
    labels={'f1_macro': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white',
    category_orders={'model': model_order[::-1]}
)
fig2.update_layout(yaxis_tickformat='.0%', showlegend=False, height=500)
show_and_save(fig2, "benchmark_f1_violin")

22:08:42 | TemporaryDirectory.cleanup() worked.
22:08:42 | shutil.rmtree worked.
22:08:42 | Chromium init'ed with kwargs {}
22:08:42 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:08:42 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpoocfs8qj.
22:08:42 | Opening browser.
22:08:42 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpl40tr9bs.
22:08:42 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpl40tr9bs
22:08:43 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpoocfs8qj/index.html
22:08:43 | Waiting on all navigates
22:08:43 | All navigates done, putting them all in queue.
22:08:43 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpoocfs8qj/index.html
22:08:43 | Waiting on all navigates
22:08:43 | All navigates done, putting them all in queue.
22:08:43 | Tab ready: A8A449A29A9C813903FEE6E4FBE4AE52
22:08:43 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_220721/plots/benchmark_f1_violin.png


## 4.2 ROC-AUC Comparison

In [14]:
# AUC + F1 Combined: Grouped Bar
metrics_long = []
for _, row in df_summary.iterrows():
    metrics_long.append({'model': row['model'], 'Metric': 'F1 Macro', 'Score': row['f1_mean'], 'Std': row['f1_std']})
    metrics_long.append({'model': row['model'], 'Metric': 'ROC-AUC', 'Score': row['auc_mean'], 'Std': row['auc_std']})
    metrics_long.append({'model': row['model'], 'Metric': 'Accuracy', 'Score': row['acc_mean'], 'Std': row['acc_std']})

ml_df = pd.DataFrame(metrics_long)

fig3 = px.bar(
    ml_df, x='model', y='Score', color='Metric', error_y='Std',
    barmode='group',
    title=f'Benchmark: All Metrics ({N_LOOPS} Runs)',
    template='plotly_white',
    text_auto='.1%',
    color_discrete_map={'F1 Macro': '#e74c3c', 'ROC-AUC': '#3498db', 'Accuracy': '#2ecc71'},
    category_orders={'model': model_order[::-1]}
)
fig3.update_layout(yaxis_tickformat='.0%', xaxis_title=None, height=500)
show_and_save(fig3, "benchmark_all_metrics")

22:08:44 | TemporaryDirectory.cleanup() worked.
22:08:44 | shutil.rmtree worked.
22:08:44 | Chromium init'ed with kwargs {}
22:08:44 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:08:44 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpksbr1p4_.
22:08:44 | Opening browser.
22:08:44 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpan32bna2.
22:08:44 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpan32bna2
22:08:44 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpksbr1p4_/index.html
22:08:44 | Waiting on all navigates
22:08:44 | All navigates done, putting them all in queue.
22:08:44 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpksbr1p4_/index.html
22:08:44 | Waiting on all navigates
22:08:45 | All navigates done, putting them all in queue.
22:08:45 | Tab ready: D83CAFB5844E8C39B913EC329C8A994F
22:08:45 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_220721/plots/benchmark_all_metrics.png


## 4.3 Per-Class F1 Comparison

In [15]:
# Per-Class F1: Heatmap
class_cols = {
    'Early (<1Y)': 'f1_early_mean',
    'Late (>1Y)': 'f1_late_mean',
    'Healthy': 'f1_healthy_mean'
}

# Build matrix
heatmap_data = []
models_sorted = df_summary.sort_values('f1_mean', ascending=False)['model'].tolist()

for model in models_sorted:
    row = df_summary[df_summary['model'] == model].iloc[0]
    heatmap_data.append([row[col] for col in class_cols.values()])

z = np.array(heatmap_data)
annot = [[f"{v:.1%}" for v in row] for row in z]

fig4 = ff.create_annotated_heatmap(
    z,
    x=list(class_cols.keys()),
    y=models_sorted,
    annotation_text=annot,
    colorscale='RdYlGn',
    showscale=True
)
fig4.update_layout(
    title='Per-Class F1 Score by Model',
    template='plotly_white',
    height=400
)
show_and_save(fig4, "benchmark_per_class_heatmap")

22:08:45 | TemporaryDirectory.cleanup() worked.
22:08:45 | shutil.rmtree worked.
22:08:45 | Chromium init'ed with kwargs {}
22:08:45 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
22:08:45 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpshes_xnl.
22:08:45 | Opening browser.
22:08:45 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp721d8c0t.
22:08:45 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp721d8c0t
22:08:45 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpshes_xnl/index.html
22:08:45 | Waiting on all navigates
22:08:45 | All navigates done, putting them all in queue.
22:08:46 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpshes_xnl/index.html
22:08:46 | Waiting on all navigates
22:08:46 | All navigates done, putting them all in queue.
22:08:46 | Tab ready: EA5569F92D41D76F5220850D03F4842C
22:08:46 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_220721/plots/benchmark_per_class_heatmap.png


# 5. Final Test Evaluation

Best model per algorithm (highest F1 on val) evaluated once on the **test set**.

In [ ]:
from sklearn.base import clone

test_results = []

# sklearn Modelle
for model_name, pipeline in MODELS.items():
    df_model = df_results[df_results['model'] == model_name]
    best_row = df_model.loc[df_model['f1_macro'].idxmax()]
    best_seed = int(best_row['seed'])

    df_bal_best = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr_best, y_tr_best = get_X_y(df_bal_best)
    clf_test = clone(MODELS[model_name])
    clf_test.fit(X_tr_best, y_tr_best)
    y_test_pred = clf_test.predict(X_test)
    y_test_proba = clf_test.predict_proba(X_test)

    f1_pc = f1_score(y_test, y_test_pred, average=None)
    test_results.append({
        'model': model_name, 'best_seed': best_seed,
        'val_f1': best_row['f1_macro'],
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'test_f1_macro': f1_score(y_test, y_test_pred, average='macro'),
        'test_roc_auc': roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro'),
        'test_f1_early': f1_pc[0], 'test_f1_late': f1_pc[1], 'test_f1_healthy': f1_pc[2],
    })
    log.info(f"\n📊 {model_name} (seed={best_seed}):")
    log.info(f"   Test: Acc={test_results[-1]['test_accuracy']:.2%} | F1={test_results[-1]['test_f1_macro']:.2%} | AUC={test_results[-1]['test_roc_auc']:.2%}")
    log.info(classification_report(y_test, y_test_pred, target_names=['Early (<1Y)', 'Late (>1Y)', 'Healthy']))

# TabPFN via subprocess
tabpfn_val_seed = int(df_results[df_results['model'] == 'TabPFN'].iloc[0]['seed'])
tabpfn_test_path = str(RESULTS_DIR / "tabpfn_test_result.json")
log.info(f"Starte TabPFN Test subprocess (seed={tabpfn_val_seed})...")
proc = subprocess.run([
    sys.executable,
    "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabpfn_test.py",
    tabpfn_test_path, str(tabpfn_val_seed), str(N_SAMPLES)
])
if proc.returncode == 0:
    tabpfn_test = json.load(open(tabpfn_test_path))
    tabpfn_test['val_f1'] = float(df_results[df_results['model'] == 'TabPFN'].iloc[0]['f1_macro'])
    test_results.append(tabpfn_test)
    log.info(f"\n📊 TabPFN (seed={tabpfn_val_seed}):")
    log.info(f"   Test: F1={tabpfn_test['test_f1_macro']:.2%} | AUC={tabpfn_test['test_roc_auc']:.2%}")
else:
    log.error("❌ TabPFN Test subprocess error")

# TabICL via subprocess
tabicl_test_path = str(RESULTS_DIR / "tabicl_test_result.json")
log.info(f"Starte TabICL Test subprocess...")
proc_icl = subprocess.run([
    sys.executable,
    "/Users/andrey/Repositories/fs-thesis/fs_thesis/run_tabicl_test.py",
    tabicl_test_path, str(42), str(N_SAMPLES)
])
if proc_icl.returncode == 0:
    tabicl_test = json.load(open(tabicl_test_path))
    tabicl_test['val_f1'] = float(df_results[df_results['model'] == 'TabICL'].iloc[0]['f1_macro'])
    test_results.append(tabicl_test)
    log.info(f"\n📊 TabICL: F1={tabicl_test['test_f1_macro']:.2%} | AUC={tabicl_test['test_roc_auc']:.2%}")
else:
    log.error("❌ TabICL Test subprocess error")

df_test_results = pd.DataFrame(test_results).sort_values('test_f1_macro', ascending=False)
df_test_results.to_csv(RESULTS_DIR / "test_final_results.csv", index=False)
log.info("\n" + df_test_results.to_string(index=False))

22:08:46 | TemporaryDirectory.cleanup() worked.
22:08:46 | shutil.rmtree worked.
22:08:47 | 
📊 DummyClassifier (seed=42):
22:08:47 |    Test: Acc=33.18% | F1=21.25% | AUC=50.07%
22:08:47 |               precision    recall  f1-score   support

 Early (<1Y)       0.05      0.33      0.08      2148
  Late (>1Y)       0.04      0.34      0.07      1630
     Healthy       0.91      0.33      0.49     40913

    accuracy                           0.33     44691
   macro avg       0.33      0.34      0.21     44691
weighted avg       0.84      0.33      0.45     44691

22:08:47 | 
📊 LogisticRegression (seed=55):
22:08:47 |    Test: Acc=58.92% | F1=37.52% | AUC=76.48%
22:08:47 |               precision    recall  f1-score   support

 Early (<1Y)       0.17      0.57      0.26      2148
  Late (>1Y)       0.08      0.57      0.13      1630
     Healthy       0.97      0.59      0.73     40913

    accuracy                           0.59     44691
   macro avg       0.40      0.58      0.38    

Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
fitting...
predicting test...


In [ ]:
# --- Test Results: Bar Chart  ---
df_test_sorted = df_test_results.sort_values('test_f1_macro')

fig5 = px.bar(
    df_test_sorted,
    x='test_f1_macro', y='model',
    orientation='h',
    text=df_test_sorted.apply(
        lambda r: f"F1={r['test_f1_macro']:.1%} | AUC={r['test_roc_auc']:.1%}", axis=1
    ),
    title='Final Test Set: F1 Macro by Model',
    labels={'test_f1_macro': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig5.update_layout(xaxis_tickformat='.0%', showlegend=False, height=400)
show_and_save(fig5, "benchmark_test_f1")

21:44:34 | Chromium init'ed with kwargs {}
21:44:34 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:44:34 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_k4rtkre.
21:44:34 | Opening browser.
21:44:34 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp4lhesuzj.
21:44:34 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp4lhesuzj
21:44:35 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_k4rtkre/index.html
21:44:35 | Waiting on all navigates
21:44:35 | All navigates done, putting them all in queue.
21:44:35 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmp_k4rtkre/index.html
21:44:35 | Waiting on all navigates
21:44:36 | All navigates done, putting them all in queue.
21:44:36 | Tab ready: 0B41904F261EE7E9F778367FB6D6DC78
21:44:36 | Getting tab from queue (has 1)
21:44:36 | Got 0B41
21:44:36 | Processing Final_T

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_213834/plots/benchmark_test_f1.png


# 6. ROC Curves (Test Set, Best Models)

One-vs-Rest ROC curves for each model on the test set, all in one plot per class.

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.base import clone
from pathlib import Path

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
class_names = ['Early (<1Y)', 'Late (>1Y)', 'Healthy']
model_colors = {
    'DummyClassifier': '#95a5a6',
    'LogisticRegression': '#3498db',
    'RandomForest': '#2ecc71',
    'XGBoost': '#e67e22',
    'TabPFN': '#e74c3c',
    'TabICL': '#9b59b6',
}

# sklearn Modelle neu trainieren
best_probas = {}
for model_name, pipeline in MODELS.items():
    df_model = df_results[df_results['model'] == model_name]
    best_seed = int(df_model.loc[df_model['f1_macro'].idxmax(), 'seed'])
    df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr, y_tr = get_X_y(df_bal)
    clf = clone(MODELS[model_name])
    clf.fit(X_tr, y_tr)
    best_probas[model_name] = clf.predict_proba(X_test)

# TabPFN proba aus Subprocess-Datei laden
tabpfn_proba_path = tabpfn_test_path.replace('.json', '_proba.npy')
if Path(tabpfn_proba_path).exists():
    best_probas['TabPFN'] = np.load(tabpfn_proba_path)
    log.info("TabPFN proba loaded")
else:
    log.warning("⚠️ TabPFN not found")

tabicl_proba_path = tabicl_test_path.replace('.json', '_proba.npy')
if Path(tabicl_proba_path).exists():
    best_probas['TabICL'] = np.load(tabicl_proba_path)
    log.info("TabICL proba loaded")
else:
    log.warning("⚠️ TabICL not found")

# Plot
from plotly.subplots import make_subplots

fig6 = make_subplots(rows=1, cols=3, subplot_titles=[f'OvR: {cn}' for cn in class_names])

for i, cn in enumerate(class_names):
    for model_name, proba in best_probas.items():
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
        roc_auc_val = auc(fpr, tpr)
        fig6.add_trace(
            go.Scatter(x=fpr, y=tpr, mode='lines',
                       name=f'{model_name} ({roc_auc_val:.3f})',
                       line=dict(color=model_colors.get(model_name, 'grey'), width=2),
                       showlegend=(i == 0)),
            row=1, col=i+1
        )
    fig6.add_trace(
        go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                   line=dict(color='grey', width=1, dash='dash'), showlegend=False),
        row=1, col=i+1
    )

fig6.update_layout(
    title='ROC Curves (Test Set) — One-vs-Rest per Class',
    template='plotly_white', height=600, width=1600,
    legend=dict(x=1.02, y=1)
)
for i in range(3):
    fig6.update_xaxes(title_text='FPR', row=1, col=i+1)
    fig6.update_yaxes(title_text='TPR', row=1, col=i+1)

show_and_save(fig6, "benchmark_roc_curves", width=1600, height=600)

21:44:36 | TemporaryDirectory.cleanup() worked.
21:44:36 | shutil.rmtree worked.
21:44:37 | TabPFN proba aus Datei geladen
21:44:37 | Chromium init'ed with kwargs {}
21:44:37 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:44:37 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpr9abm1_y.
21:44:37 | Opening browser.
21:44:37 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpbb0uqdlr.
21:44:37 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpbb0uqdlr
21:44:38 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpr9abm1_y/index.html
21:44:38 | Waiting on all navigates
21:44:38 | All navigates done, putting them all in queue.
21:44:38 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpr9abm1_y/index.html
21:44:38 | Waiting on all navigates
21:44:39 | All navigates done, putting them all in queue.
21:44:39 | Tab ready: 2

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_213834/plots/benchmark_roc_curves.png


# 7. Data Quality Check: Dead but "Healthy"?

Patients classified as **target=2 (healthy/censored)** who have a recorded date of death (`dod`). These patients died but were never diagnosed with heart failure — they are **correctly** censored (not false labels), but it's important to know how many there are and whether they bias the model.

In [ ]:
# ── Data Quality: Deceased Patients in the "Healthy" Class ──
import polars as pl

# df is the full dataset (before the split)
df_quality = df.to_pandas() if hasattr(df, 'to_pandas') else df

# Patients with target=2 (healthy/censored) AND date of death (dod) present
dead_but_healthy = df.filter(
    (pl.col("target") == 2) & (pl.col("dod").is_not_null())
)

total_healthy = df.filter(pl.col("target") == 2).height
n_dead_healthy = dead_but_healthy.height

log.info(f"Total patients with target=2 (healthy/censored): {total_healthy:,}")
log.info(f"With date of death (dod): {n_dead_healthy:,} ({n_dead_healthy/total_healthy:.1%})")
log.info(f"Without dod (truly alive): {total_healthy - n_dead_healthy:,}")

# Distribution of survival time for the 'dead healthy' patients
if n_dead_healthy > 0:
    t_death_stats = dead_but_healthy.select("t_death").to_pandas()["t_death"].describe()
    log.info(f"\nSurvival time (t_death) of deceased 'healthy' patients:")
    log.info(t_death_stats)
    
    fig_dq = px.histogram(
        dead_but_healthy.select("t_death").to_pandas(), 
        x="t_death", nbins=50,
        title=f"Deceased patients without HF diagnosis (n={n_dead_healthy:,}): Days until death",
        labels={"t_death": "Days from baseline to death"},
        template="plotly_white"
    )
    fig_dq.add_vline(x=365, line_dash="dash", line_color="red", annotation_text="1 year")
    show_and_save(fig_dq, "data_quality_dead_but_healthy")

21:44:39 | Total patients with target=2 (healthy/censored): 204,562
21:44:39 | With date of death (dod): 29,503 (14.4%)
21:44:39 | Without dod (truly alive): 175,059
21:44:39 | Chromium init'ed with kwargs {}
21:44:39 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:44:39 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpb04jf_f6.
21:44:39 | Opening browser.
21:44:39 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpl3d7nl0t.
21:44:39 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpl3d7nl0t



Survival time (t_death) of deceased 'healthy' patients:
count    29503.000000
mean       651.097956
std        966.801290
min      -2520.000000
25%         39.000000
50%        217.000000
75%        845.000000
max       5615.000000
Name: t_death, dtype: float64


21:44:40 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpb04jf_f6/index.html
21:44:40 | Waiting on all navigates
21:44:40 | All navigates done, putting them all in queue.
21:44:40 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpb04jf_f6/index.html
21:44:40 | Waiting on all navigates
21:44:41 | All navigates done, putting them all in queue.
21:44:41 | Tab ready: A9A09578453877FC799FEBD58A583BCB
21:44:41 | Getting tab from queue (has 1)
21:44:41 | Got A9A0
21:44:41 | Processing Deceased_patients_without_HF_diagnosis_n29503_Days_until_death.png
21:44:41 | Sending big command for Deceased_patients_without_HF_diagnosis_n29503_Days_until_death.png.
21:44:41 | Sent big command for Deceased_patients_without_HF_diagnosis_n29503_Days_until_death.png.
21:44:41 | Reloading tab A9A0 before return.
21:44:41 | Putting tab A9A0 back (queue size: 0).
21:44:41 | Waiting for all cleanups to finish.
21:44:41 | Exiting Kaleido
21:44:41 | Closing bro

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_213834/plots/data_quality_dead_but_healthy.png


# 8. Correlation vs. Causality (Feature Analysis)

**Philipp's feedback: "Remove confounders"** — Identify features with correlation but no causality.

| Analysis | What it measures | Method |
|----------|------------------|--------|
| **Correlation** (univariate) | How strongly does a feature *alone* relate to the target? | Cramér's V (categorical), Eta² (numerical) |
| **Predictive importance** (multivariate) | How much *unique* predictive power does a feature have, when all others are known? | Permutation Importance (F1 Macro) |

**Interpretation:**
- High correlation + high importance → **True driver** (e.g., age, BMI)
- High correlation + low/negative importance → **Confounder** (e.g., insurance correlates with age)
- Low correlation + low importance → **Irrelevant** (can be removed)


## 8.1 Univariate Correlation (Feature ↔ Target)

In [ ]:
# Univariate Correlation: How strongly does each feature alone relate to the target?
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    """Cramér's V: Association for categorical × categorical (0 = none, 1 = perfect)."""
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = len(x)
    min_dim = min(ct.shape) - 1
    if min_dim == 0 or n == 0:
        return 0.0
    return np.sqrt(chi2 / (n * min_dim))

def eta_squared(feature_values, target_values):
    """Eta²: Effect size for numerical × categorical (ANOVA). 0 = none, 1 = perfect."""
    groups = [feature_values[target_values == c].dropna() for c in sorted(target_values.unique())]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return 0.0
   
    grand_mean = feature_values.dropna().mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = ((feature_values.dropna() - grand_mean)**2).sum()
    if ss_total == 0:
        return 0.0
    return ss_between / ss_total

# Compute correlation for each feature
df_val_pd = X_val.copy()
df_val_pd['target'] = y_val

correlation_results = []
for feature in FEATURE_COLS:
    if feature in CAT_COLS:
        corr = cramers_v(df_val_pd[feature], df_val_pd['target'])
        method = "Cramér's V"
    else:
        corr = eta_squared(df_val_pd[feature], df_val_pd['target'])
        method = "Eta²"
    correlation_results.append({
        'Feature': feature,
        'Correlation': corr,
        'Method': method,
        'Type': 'categorical' if feature in CAT_COLS else 'numerical'
    })

df_corr = pd.DataFrame(correlation_results).sort_values('Correlation', ascending=False)

log.info("\nUnivariate Correlation (Feature → Target):\n")
log.info(df_corr.to_string(index=False, float_format='{:.4f}'.format))

# Plot
fig_corr = px.bar(
    df_corr.sort_values('Correlation'),
    x='Correlation', y='Feature', orientation='h',
    color='Type',
    color_discrete_map={'categorical': '#3498db', 'numerical': '#e74c3c'},
    text=df_corr.sort_values('Correlation').apply(
        lambda r: f"{r['Correlation']:.3f} ({r['Method']})", axis=1
    ),
    title=f"Univariate Correlation: Feature ↔ Target (n={len(y_val)})",
    labels={'Correlation': 'Association Strength', 'Feature': ''},
    template='plotly_white'
)
fig_corr.update_layout(height=450)
show_and_save(fig_corr, "correlation_univariate")

21:44:41 | Chromium init'ed with kwargs {}
21:44:41 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:44:41 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpoirttjio.
21:44:41 | Opening browser.
21:44:41 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpe_l1vp6q.
21:44:41 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpe_l1vp6q



Univariate Correlation (Feature → Target):

       Feature  Correlation     Method        Type
admission_type       0.1408 Cramér's V categorical
     insurance       0.1237 Cramér's V categorical
          race       0.0798 Cramér's V categorical
marital_status       0.0747 Cramér's V categorical
      language       0.0582 Cramér's V categorical
    anchor_age       0.0482       Eta²   numerical
        gender       0.0416 Cramér's V categorical
           bmi       0.0065       Eta²   numerical


21:44:41 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpoirttjio/index.html
21:44:41 | Waiting on all navigates
21:44:41 | All navigates done, putting them all in queue.
21:44:42 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpoirttjio/index.html
21:44:42 | Waiting on all navigates
21:44:42 | All navigates done, putting them all in queue.
21:44:42 | Tab ready: 547CF5A8950FA31712905280166929B2
21:44:42 | Getting tab from queue (has 1)
21:44:42 | Got 547C
21:44:42 | Processing Univariate_Correlation_Feature__Target_n35753.png
21:44:42 | Sending big command for Univariate_Correlation_Feature__Target_n35753.png.
21:44:42 | Sent big command for Univariate_Correlation_Feature__Target_n35753.png.
21:44:42 | Reloading tab 547C before return.
21:44:42 | Putting tab 547C back (queue size: 0).
21:44:42 | Waiting for all cleanups to finish.
21:44:42 | Exiting Kaleido
21:44:42 | Closing browser.
21:44:42 | Closing browser.
21:44:42 | Tempor

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_213834/plots/correlation_univariate.png


## 8.2 Predictive Importance (Permutation Importance)

Measures how much **unique** predictive power a feature has **when all other features are known**. A feature with high correlation but low importance is a confounder — it only correlates because it is associated with a true driver.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.base import clone

importance_results = []

for model_name in MODELS.keys():
    log.info(f"\n🔄 {model_name}...")
    df_model = df_results[df_results['model'] == model_name]
    best_seed = int(df_model.loc[df_model['f1_macro'].idxmax(), 'seed'])

    df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr, y_tr = get_X_y(df_bal)
    clf = clone(MODELS[model_name])
    clf.fit(X_tr, y_tr)
    perm = permutation_importance(clf, X_val, y_val, n_repeats=10,
                                   random_state=42, scoring='f1_macro', n_jobs=-1)
    for j, feature in enumerate(FEATURE_COLS):
        importance_results.append({
            'Model': model_name, 'Feature': feature,
            'Importance': perm.importances_mean[j],
            'Std': perm.importances_std[j],
        })
    log.info(f"  ✅ done")


log.info("TabPFN Permutation Importance skipped (35k Val × 10 repeats)")
for feature in FEATURE_COLS:
    importance_results.append({
        'Model': 'TabPFN', 'Feature': feature,
        'Importance': float('nan'), 'Std': float('nan')
    })

log.info("TabICL Permutation Importance skipped")
for feature in FEATURE_COLS:
    importance_results.append({
        'Model': 'TabICL', 'Feature': feature,
        'Importance': float('nan'), 'Std': float('nan')
    })

df_importance = pd.DataFrame(importance_results)
df_importance.to_csv(RESULTS_DIR / "permutation_importance_all_models.csv", index=False)

df_imp_avg = df_importance[
    ~df_importance['Model'].isin(['DummyClassifier', 'TabPFN'])
].groupby('Feature').agg(
    Importance_mean=('Importance', 'mean'),
    Importance_std=('Importance', 'std'),
).reset_index().sort_values('Importance_mean', ascending=False)

log.info("\n📊 Average Permutation Importance (excluding Dummy + TabPFN):\n")
log.info(df_imp_avg.to_string(index=False, float_format='{:.4f}'.format))


🔄 DummyClassifier...
  ✅ done

🔄 LogisticRegression...


21:44:47 | TemporaryDirectory.cleanup() worked.
21:44:47 | shutil.rmtree worked.


  ✅ done

🔄 RandomForest...
  ✅ done

🔄 XGBoost...


21:45:00 | TabPFN Permutation Importance übersprungen (35k Val × 10 repeats)


  ✅ done

📊 Average Permutation Importance (excluding Dummy + TabPFN):

       Feature  Importance_mean  Importance_std
    anchor_age           0.0352          0.0168
admission_type           0.0310          0.0055
           bmi           0.0155          0.0108
     insurance           0.0019          0.0048
          race           0.0018          0.0030
        gender           0.0006          0.0003
      language           0.0001          0.0015
marital_status          -0.0003          0.0014


In [ ]:
# ── Importance Heatmap: pro Modell × Feature ──
pivot = df_importance.pivot_table(index='Model', columns='Feature', values='Importance')
models_order = ['DummyClassifier', 'LogisticRegression', 'RandomForest', 'XGBoost', 'TabPFN', 'TabICL']
pivot = pivot.reindex(models_order)
features_order = df_imp_avg.sort_values('Importance_mean', ascending=True)['Feature'].tolist()
pivot = pivot[features_order]

annot = [[f"{v:.3f}" for v in row] for row in pivot.values]

fig_imp_heat = ff.create_annotated_heatmap(
    pivot.values, x=features_order, y=models_order,
    annotation_text=annot, colorscale='RdBu', reversescale=False, showscale=True
)
fig_imp_heat.update_layout(
    title='Permutation Importance: Feature × Model (F1 Macro)',
    template='plotly_white', height=400
)
show_and_save(fig_imp_heat, "importance_heatmap_all_models")

21:45:00 | Chromium init'ed with kwargs {}
21:45:00 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:45:00 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpol9q_91q.
21:45:00 | Opening browser.
21:45:00 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpbjmwda9j.
21:45:00 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpbjmwda9j
21:45:00 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpol9q_91q/index.html
21:45:00 | Waiting on all navigates
21:45:00 | All navigates done, putting them all in queue.
21:45:01 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpol9q_91q/index.html
21:45:01 | Waiting on all navigates
21:45:01 | All navigates done, putting them all in queue.
21:45:01 | Tab ready: 02EF68FF6D21434867780752D2C47696
21:45:01 | Getting tab from queue (has 1)
21:45:01 | Got 02EF
21:45:01 | Processing Permuta

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_213834/plots/importance_heatmap_all_models.png


## 8.3 Correlation vs. Causality — The Confounder Plot

Compares univariate correlation (descriptive) with multivariate importance (predictive). Features in the **upper left quadrant** (high correlation, low importance) are "confounders" — they correlate because they are proxies for true drivers.

In [ ]:
# --- Correlation vs. Causality: Scatter Plot ---
df_combined = df_corr.merge(df_imp_avg, on='Feature')

# Feature classification
def classify_feature(row):
    corr_threshold = df_combined['Correlation'].median()
    imp_threshold = 0.005  # minimal positive importance
    if row['Correlation'] >= corr_threshold and row['Importance_mean'] >= imp_threshold:
        return '✅ True driver'
    elif row['Correlation'] >= corr_threshold and row['Importance_mean'] < imp_threshold:
        return '⚠️ Confounder'
    elif row['Correlation'] < corr_threshold and row['Importance_mean'] >= imp_threshold:
        return '🔍 Hidden driver'
    else:
        return '❌ Irrelevant'

df_combined['Category'] = df_combined.apply(classify_feature, axis=1)

# Scatter: Correlation (x) vs. Importance (y)
fig_scatter = px.scatter(
    df_combined, 
    x='Correlation', y='Importance_mean',
    text='Feature',
    color='Category',
    color_discrete_map={
        '✅ True driver': '#2ecc71',
        '⚠️ Confounder': '#e67e22',
        '🔍 Hidden driver': '#3498db',
        '❌ Irrelevant': '#95a5a6',
    },
    error_y='Importance_std',
    title='Correlation vs. Predictive Importance — "Confounder Plot"',
    labels={
        'Correlation': 'Univariate Correlation (Cramér\'s V / Eta²)',
        'Importance_mean': 'Permutation Importance (mean across models, F1 Macro)'
    },
    template='plotly_white'
)

# Quadrant lines
corr_median = df_combined['Correlation'].median()
fig_scatter.add_hline(y=0.005, line_dash="dash", line_color="grey", opacity=0.5,
                       annotation_text="Importance threshold")
fig_scatter.add_vline(x=corr_median, line_dash="dash", line_color="grey", opacity=0.5,
                       annotation_text="Correlation median")
fig_scatter.add_hline(y=0, line_dash="solid", line_color="black", opacity=0.3)

fig_scatter.update_traces(textposition='top center', marker=dict(size=14))
fig_scatter.update_layout(height=600, width=900)
show_and_save(fig_scatter, "correlation_vs_causality_scatter", width=900, height=600)


21:45:01 | TemporaryDirectory.cleanup() worked.
21:45:01 | shutil.rmtree worked.
21:45:01 | TemporaryDirectory.cleanup() worked.
21:45:01 | shutil.rmtree worked.
21:45:01 | Chromium init'ed with kwargs {}
21:45:01 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:45:01 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpdeurhngb.
21:45:01 | Opening browser.
21:45:01 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpl_m63jub.
21:45:01 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpl_m63jub
21:45:02 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpdeurhngb/index.html
21:45:02 | Waiting on all navigates
21:45:02 | All navigates done, putting them all in queue.
21:45:02 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpdeurhngb/index.html
21:45:02 | Waiting on all navigates
21:45:03 | All navigates done, putting the

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_213834/plots/correlation_vs_causality_scatter.png


In [ ]:
df_side = df_combined.sort_values('Correlation', ascending=True)

fig_side = go.Figure()
fig_side.add_trace(go.Bar(
    y=df_side['Feature'], x=df_side['Correlation'],
    name='Correlation (univariate)', orientation='h',
    marker_color='#3498db', opacity=0.8
))
fig_side.add_trace(go.Bar(
    y=df_side['Feature'], x=df_side['Importance_mean'],
    name='Importance (multivariate)', orientation='h',
    marker_color='#e74c3c', opacity=0.8,
    error_x=dict(type='data', array=df_side['Importance_std'].values, visible=True)
))
fig_side.update_layout(
    barmode='group',
    title='Correlation vs. Predictive Importance (Side-by-Side)',
    xaxis_title='Strength',
    yaxis_title=None,
    template='plotly_white',
    height=500,
    legend=dict(x=0.6, y=0.05)
)
show_and_save(fig_side, "correlation_vs_importance_bars")

log.info("\n📊 Feature Classification:\n")
display_df = df_combined[['Feature', 'Correlation', 'Method', 'Importance_mean', 'Importance_std', 'Category']]
display_df = display_df.sort_values('Correlation', ascending=False)
log.info(display_df.to_string(index=False, float_format='{:.4f}'.format))

df_combined.to_csv(RESULTS_DIR / "correlation_vs_importance.csv", index=False)

log.info("\n" + "="*60)
true_drivers = df_combined[df_combined['Category'].str.contains('True driver')]
confounders  = df_combined[df_combined['Category'].str.contains('Confounder')]
irrelevant   = df_combined[df_combined['Category'].str.contains('Irrelevant')]

if not true_drivers.empty:
    log.info(f"✅ True drivers: {', '.join(true_drivers['Feature'].tolist())}")
if not confounders.empty:
    log.info(f"⚠️  Confounders: {', '.join(confounders['Feature'].tolist())}")
    log.info(f"   → Correlate with target but no unique predictive power.")
if not irrelevant.empty:
    log.info(f"❌ Irrelevant: {', '.join(irrelevant['Feature'].tolist())}")
log.info("="*60)

21:45:03 | TemporaryDirectory.cleanup() worked.
21:45:03 | shutil.rmtree worked.
21:45:03 | Chromium init'ed with kwargs {}
21:45:03 | Found chromium path: /Applications/Brave Browser.app/Contents/MacOS/Brave Browser
21:45:03 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpkm4vi81v.
21:45:03 | Opening browser.
21:45:03 | Temp directory created: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmph_b246p0.
21:45:03 | Temporary directory at: /var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmph_b246p0
21:45:03 | Conforming 0 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpkm4vi81v/index.html
21:45:03 | Waiting on all navigates
21:45:03 | All navigates done, putting them all in queue.
21:45:03 | Conforming 1 to file:///var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/tmpkm4vi81v/index.html
21:45:03 | Waiting on all navigates
21:45:04 | All navigates done, putting them all in queue.
21:45:04 | Tab ready: 8E601C1C87F8E9E077F93E427097FF2C
21:45:04 |

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260309_213834/plots/correlation_vs_importance_bars.png



📊 Feature Classification:

       Feature  Correlation     Method  Importance_mean  Importance_std        Category
admission_type       0.1408 Cramér's V           0.0310          0.0055   ✅ True driver
     insurance       0.1237 Cramér's V           0.0019          0.0048   ⚠️ Confounder
          race       0.0798 Cramér's V           0.0018          0.0030   ⚠️ Confounder
marital_status       0.0747 Cramér's V          -0.0003          0.0014   ⚠️ Confounder
      language       0.0582 Cramér's V           0.0001          0.0015    ❌ Irrelevant
    anchor_age       0.0482       Eta²           0.0352          0.0168 🔍 Hidden driver
        gender       0.0416 Cramér's V           0.0006          0.0003    ❌ Irrelevant
           bmi       0.0065       Eta²           0.0155          0.0108 🔍 Hidden driver

✅ True drivers: admission_type
⚠️  Confounders: insurance, race, marital_status
   → Correlate with target but no unique predictive power.
❌ Irrelevant: language, gender


# Summary

This notebook provides four key results for the thesis:

1. **Benchmark** (Section 3-6): TabPFN compared against DummyClassifier, LogisticRegression, RandomForest, and XGBoost — all with identical data pipeline, 20 runs each.

2. **Data Quality** (Section 7): Quantifies how many target=2 patients are actually deceased (censored correctly, but important for interpretation).

3. **Korrelation vs. Kausalität** (Section 8.1–8.3): Systematic analysis distinguishing:
   - **Univariate Korrelation** (Cramér's V / Eta²): deskriptive Assoziation
   - **Permutation Importance** (F1 Macro, alle Modelle): prädiktive Bedeutung
   - **Hosenträger-Plot**: Identifiziert Confounder (hohe Korrelation, keine eigene Vorhersagekraft)


All results are saved in the run folder for reproducibility.